In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"urzadzenie: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_id)

print("ładowanie modelu")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    low_cpu_mem_usage=True
).to(device)

model.eval()
print("llm ready")

urzadzenie: mps
ładowanie modelu


Loading weights: 100%|██████████| 338/338 [00:07<00:00, 47.72it/s]


llm ready


In [9]:
messages = [
    {
        "role": "system", 
        "content": "You are a helpful, direct, and concise AI assistant."
    },
    {
        "role": "user", 
        "content": "Tell me a joke with a twist?"
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
input_length = inputs.input_ids.shape[1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=150, 
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1, 
        pad_token_id=tokenizer.eos_token_id
    )

response_tokens = output_ids[0][input_length:]
odpowiedz = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()

print(odpowiedz)

Why don't scientists trust atoms anymore?

Because they make up everything!


In [ ]:
import json
import os
import pandas as pd
import torch
from tqdm.auto import tqdm

test_dataset = [
    {
        "id": 1,
        "category": "high_entropy",
        "question": (
            "What are the main arguments for and against space exploration?"
        ),
    },
    {
        "id": 2,
        "category": "high_entropy",
        "question": (
            "Describe the psychological benefits of spending time in nature."
        ),
    },
    {
        "id": 3,
        "category": "high_entropy",
        "question": (
            "Explain how the invention of the printing press changed European"
            " society."
        ),
    },
    {
        "id": 4,
        "category": "high_entropy",
        "question": (
            "Why do different cultures have different musical scales and"
            " preferences?"
        ),
    },
    {
        "id": 5,
        "category": "high_entropy",
        "question": (
            "Explain the concept of cognitive bias and give two real-life"
            " examples."
        ),
    },
    {
        "id": 6,
        "category": "low_entropy",
        "question": (
            "In which year did the Apollo 11 mission land on the Moon and who"
            " was the pilot?"
        ),
    },
    {
        "id": 7,
        "category": "low_entropy",
        "question": "What is Newton's third law of motion? State it clearly.",
    },
    {
        "id": 8,
        "category": "low_entropy",
        "question": (
            "Who was the first emperor of Rome and in what year did his rule"
            " begin?"
        ),
    },
    {
        "id": 9,
        "category": "low_entropy",
        "question": (
            "Define what a primary key is in a relational SQL database."
        ),
    },
    {
        "id": 10,
        "category": "low_entropy",
        "question": (
            "What is the chemical formula for water and table salt (sodium"
            " chloride)?"
        ),
    },
]

results = []
print(f"Rozpoczynam generowanie odpowiedzi dla {len(test_dataset)} pytań...")

for item in tqdm(test_dataset):
  messages = [
      {
          "role": "system",
          "content": (
              "You are a helpful, direct, and concise AI assistant. Answer"
              " clearly in 2-3 sentences."
          ),
      },
      {"role": "user", "content": item["question"]},
  ]

  formatted_prompt = tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )

  inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
  input_len = inputs.input_ids.shape[1]

  with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

  response_tokens = output_ids[0][input_len:]
  response_text = tokenizer.decode(
      response_tokens, skip_special_tokens=True
  ).strip()

  results.append({
      "id": item["id"],
      "category": item["category"],
      "question": item["question"],
      "response": response_text,
  })

with open("wyniki_generacji.json", "w", encoding="utf-8") as f:
  json.dump(results, f, ensure_ascii=False, indent=2)

df = pd.DataFrame(results)
df.to_csv("wyniki_generacji.csv", index=False, encoding="utf-8")

print("\n Zapisano pomyślnie do 'wyniki_generacji.json' oraz 'wyniki_generacji.csv'!")

Rozpoczynam generowanie odpowiedzi dla 10 pytań...


100%|██████████| 10/10 [00:30<00:00,  3.08s/it]



 Zapisano pomyślnie do 'wyniki_generacji.json' oraz 'wyniki_generacji.csv'!
